In [4]:

import os
import re
import warnings
import random
from collections import defaultdict
from typing import Dict, List, Tuple

import torch
import torch.nn.functional as F
import numpy as np
import numpy as np
import torch
from tqdm.notebook import tqdm
from transformers import GPT2LMHeadModel, GPT2Tokenizer

warnings.filterwarnings("ignore")

In [5]:
def seed_everything(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything(42)

1) Реализованы методы `greedy_sampling` и `generate`
2) Реализован метод `random_sampling` и поддержан в `generate`
3) Реализован метод `_beam_search_generate` и поддержан в `generate`
4) Реализованы методы `apply_top_p`, `apply_top_k`, `apply_temperature` и поддержаны их в `generate`  

In [21]:
class Model:
    def __init__(self, model_name: str = "gpt2"):
        self.model = GPT2LMHeadModel.from_pretrained(model_name)
        self.tokenizer = GPT2Tokenizer.from_pretrained(model_name)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.vocab_size = self.tokenizer.vocab_size

    def greedy_sampling(self, logits: torch.Tensor) -> int:
        return torch.argmax(logits, dim=-1).item()

    def random_sampling(self, logits: torch.Tensor) -> int:
        probs = F.softmax(logits, dim=-1)
        return torch.multinomial(probs, 1).item()

    def _beam_search_generate(
        self,
        prompt: str,
        max_length: int,
        num_beams: int) -> str:
        # Токенизация
        input_ids = self.tokenizer.encode(prompt, return_tensors="pt")
        
        # лучи
        beam_scores = torch.zeros(num_beams, device=input_ids.device)
        beam_sequences = input_ids.repeat(num_beams, 1)
        
        # Генерация токенов
        for _ in range(max_length - len(input_ids[0])):
            # логиты для всех лучей
            with torch.no_grad():
                outputs = self.model(beam_sequences)
                next_token_logits = outputs.logits[:, -1, :]
            
            
            next_token_probs = F.log_softmax(next_token_logits, dim=-1)
            
            # скоры
            vocab_size = next_token_probs.size(-1)
            exp_scores = beam_scores.unsqueeze(-1) + next_token_probs
            


            # top-k кандидатов
            top_sc, top_ind = torch.topk(exp_scores.view(-1), num_beams)

            
            beam_ind = top_ind // vocab_size
            token_ind = top_ind % vocab_size
            
            new_seq = []
            for i in range(num_beams):
                beam_idx = beam_ind[i]
                token_idx = token_ind[i]
                new_seq.append(torch.cat([beam_sequences[beam_idx], token_idx.unsqueeze(0)]))
            
            beam_seq = torch.stack(new_seq)
            beam_sc = top_sc
           
            if all(self.tokenizer.eos_token_id in seq for seq in beam_seq):
                break
        
        
        best_seq = beam_seq[torch.argmax(beam_sc)]
        
        return self.tokenizer.decode(best_seq, skip_special_tokens=True)

    def apply_temperature(self, logits, temperature = 1.0) -> torch.Tensor:
        return logits / temperature

    def apply_top_k(self, logits: torch.Tensor, top_k: int = None) -> torch.Tensor:
        if top_k is None or top_k <= 0 or top_k > self.vocab_size:
            return logits
        
        values, _ = torch.topk(logits, top_k)
        min_val = values[:, -1] if logits.dim() > 1 else values[-1]
        mask = logits < min_val.unsqueeze(-1) if logits.dim() > 1 else logits < min_val


    
        filtered_logits = logits.masked_fill(mask, -float('inf'))
        
        return filtered_logits

    def apply_top_p(self, logits: torch.Tensor, top_p: float = 1.0) -> torch.Tensor:
        if top_p >= 1.0:
            return logits
       
        sorted_logits, sorted_indices = torch.sort(logits, descending=True)
        sorted_probs = F.softmax(sorted_logits, dim=-1)
        
        # кумулятивные вероятности
        cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
        
        # токены с кумулятивной вероятностью выше top_p - удаляются
        sorted_remove = cumulative_probs > top_p
        
        # Сдвиг на один, чтобы сохранить первый токен, который превышает top_p
        sorted_remove[..., 1:] = sorted_remove[..., :-1].clone()
        sorted_remove[..., 0] = 0
        
        # Восстанавение оригинального порядка и обнуление ненужных логитов
        indices_remove = sorted_remove.scatter( -1, sorted_indices, sorted_remove)
        filtered_logits = logits.masked_fill(indices_remove, -float('inf'))
        
        return filtered_logits

    def generate(self, prompt: str, max_length: int = 50, strategy: str = "greedy",
        temperature: float = 1.0, top_k: int = 0, top_p: float = 1.0, num_beams= 3 ) -> str:
        
        if strategy == "beam_search":
            return self._beam_search_generate(prompt, max_length, num_beams)
        
        input_ids = self.tokenizer.encode(prompt, return_tensors="pt")
        generated = input_ids.clone()
        

        for _ in range(max_length - len(input_ids[0])):
            with torch.no_grad():
                outputs = self.model(generated)
                next_token_logits = outputs.logits[:, -1, :]
            
            # температурное scaling
            if temperature != 1.0:
                next_token_logits = self.apply_temperature(next_token_logits, temperature)
            
            # top-k фильтрация
            if top_k > 0:
                next_token_logits = self.apply_top_k(next_token_logits, top_k)
            
            # top-p (nucleus) фильтрация
            if top_p < 1.0:
                next_token_logits = self.apply_top_p(next_token_logits, top_p)
            
            # следующий токен в зависимости от стратегии
            if strategy == "greedy":
                next_token = self.greedy_sampling(next_token_logits)
            elif strategy == "random":
                next_token = self.random_sampling(next_token_logits)
            
            generated = torch.cat([generated, torch.tensor([[next_token]], device=generated.device)], dim=1)
            
            # если достигли конца последовательности - завершение
            if next_token == self.tokenizer.eos_token_id:
                break
        
        return self.tokenizer.decode(generated[0], skip_special_tokens=True)


In [26]:
# Демонстрация работы с различными параметрами

model = Model()
prompt = "The future of artificial intelligence industry"
max_l = 60

print("===Greedy Sampling===")
result1 = model.generate(prompt, strategy="greedy", max_length=max_l)
print(result1)
print()

print("===Random Sampling===")
result2 = model.generate(prompt, strategy="random", max_length=max_l)
print(result2)
print()

print("===Random Sampling with Temperature===")
result3 = model.generate(prompt, strategy="random", temperature=1.5, max_length=max_l)
print(result3)
print()

print("===Random Sampling with Top-k===")
result4 = model.generate(prompt, strategy="random", top_k=10, max_length=max_l)
print(result4)
print()

print("===Random Sampling with Top-p===")
result5 = model.generate(prompt, strategy="random", top_p=0.9, max_length=max_l)
print(result5)
print()

print("===Beam Search===")
result6 = model.generate(prompt, strategy="beam_search", num_beams=3, max_length=max_l)
print(result6)

===Greedy Sampling===
The future of artificial intelligence industry is uncertain.

"We're not sure what the future of artificial intelligence industry is," said John D. Schulman, a professor of computer science at the University of California, Berkeley. "We're not sure what the future of artificial intelligence industry is."

===Random Sampling===
The future of artificial intelligence industry: don't you know it janky 40 piece D multi frame commercial trailer 5 photos on the side Brooklyn Boulevard in partnership with Material Design 2 that mark the track record of immersive living.

Lowest pitched video: Joe Odumini

As late as 2013

===Random Sampling with Temperature===
The future of artificial intelligence industry Barrier Vaccinis servers 'Remove Robot Stock Usernames Built above computers vulnerability benefits orders combined USD 100.29 Valid One *.Examples 271 Stna366 Coral Driver Manager thrust crab UntilApril 1st he left plan Father after 78ilia genentoso mobile soleenge 13 

Как видно, самая качественная генерация у top-k и top-p стратегий (речь вполне связная и содержательная), в виду того, что в отличие от тех же детерменированных методов (Greedy/Beam, которые выбирают лишь самые вероятные последовательности, потму их речь не содержательно, хоть вроде и +- логична (последнее для Greedy применимо)), этим методам предоставлен выбор не самой высоковерояной последовательности слов в продолжении, а есть возможность подбирать уместные синонимы, делая речь разнообразнее (думаю, если бы top_k сделать бы больше и также увеличить параметр top-p, то речь была бы интерснее, не потеряв связности). Что касается метода с приминением температуры распределения, то здесь потеряли связь предложения (нужно для улучшения корректировать температуру), второй метод обычного ничем не ограниченного рандомного сэмплинга кажется лично мне вообще не эффективным для длинной генерации.